# RAG sobre Historia de las Guerras Mundiales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhonattanreales21/nlp_icesi/blob/main/unidad_6/Cano_Reales_RAG.ipynb)

**Autores:** Andrés Cano & Jhonattan Reales  
**Curso:** Procesamiento de Lenguaje Natural — Maestría MIAA, ICESI  
**Unidad 6:** Retrieval-Augmented Generation (RAG)

---

Objetivo
----

En este taller construimos un **chatbot conversacional basado en RAG** especializado en la historia de las Guerras Mundiales. A diferencia de un LLM estándar, nuestro sistema recupera fragmentos relevantes de una base de conocimiento propia (artículos de Wikipedia en inglés) antes de generar cada respuesta, lo que garantiza mayor precisión y trazabilidad de las fuentes.

**¿Por qué RAG en lugar de un LLM estándar?**

1. **Alucinaciones reducidas**: el LLM sólo puede responder con lo que está en el corpus recuperado.
2. **Trazabilidad**: cada respuesta incluye las fuentes (artículos de Wikipedia) que la fundamentan.
3. **Conocimiento actualizable**: basta con re-indexar documentos nuevos sin reentrenar el modelo.
4. **Eficiencia**: usar un modelo pequeño (7B) con un buen retriever supera a un modelo grande sin contexto.

Componentes del sistema
----

El sistema fue construido seleccionando herramientas que balancean rendimiento, reproducibilidad y ejecución local, alineadas con el objetivo de crear un chatbot RAG especializado en historia de las Guerras Mundiales.

- **Dataset:** `wikimedia/wikipedia` (filtrado por palabras claves de WW1/WW2) — Artículos de Wikipedia en inglés sobre ambas guerras mundiales, sirviendo como base de conocimiento verificable y trazable para las respuestas del chatbot.
- **LLM:** `mistral:7b-instruct` (Ollama) — Modelo de 7B parámetros ejecutado localmente; ofrece mejor capacidad de razonamiento e instrucción que alternativas más ligeras como `llama3.2:3b`.
- **Embeddings:** `intfloat/multilingual-e5-large` — Modelo de embeddings multilingüe con alto rendimiento en inglés, usado para representar semánticamente los fragmentos recuperados.
- **Vector Store:** FAISS — Índice vectorial local que permite búsqueda eficiente por similitud sobre los fragmentos de Wikipedia indexados.
- **Framework:** LangChain + Gradio — LangChain orquesta la cadena RAG (recuperación → aumentación → generación), mientras Gradio expone una interfaz conversacional interactiva para el usuario.

## Paquetes y Setup

In [2]:
import warnings

warnings.filterwarnings("ignore")

# Verificar si se está ejecutando en Google Colab
try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Ejecutando en Colab: {IN_COLAB}")

Ejecutando en Colab: False


In [3]:
# Instalamos las dependencias según el entorno de ejecución
# En Colab usamos FAISS con GPU; en local, la variante CPU por compatibilidad
if IN_COLAB:
    !pip install -q \
        langchain langchain-ollama langchain-community \
        langchain-huggingface langchain-text-splitters \
        faiss-gpu-cu12 sentence-transformers \
        datasets gradio wordcloud \
        matplotlib seaborn nltk colab-xterm
else:
    !pip install -r requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from wordcloud import WordCloud, STOPWORDS
import nltk

from langchain_text_splitters import RecursiveCharacterTextSplitter

---
# 1. Carga del Dataset

Usamos **`wikimedia/wikipedia`** (dump 20231101 en inglés) cargado en modo **streaming** para evitar descargar el dataset completo (~20 GB). Filtramos artículos cuyos títulos contengan palabras clave relacionadas con la Primera y Segunda Guerra Mundial.

Palabras clave de filtrado
----

Cubrimos eventos, lugares, organizaciones y tecnologías representativas de ambas guerras para obtener un corpus temáticamente rico y equilibrado.

In [5]:
from datasets import load_dataset
import pandas as pd

# Palabras clave que definen nuestro corpus de Guerras Mundiales
WW_KEYWORDS = [
    # General
    "World War",
    # WWI específico
    "Western Front",
    "Eastern Front",
    "Trench warfare",
    "Armistice",
    "Treaty of Versailles",
    "Gallipoli",
    "Somme",
    "Verdun",
    "Austro-Hungarian",
    "Ottoman Empire",
    "Archduke Franz Ferdinand",
    "Sarajevo assassination",
    "U-boat",
    "submarine warfare",
    "chemical warfare",
    "mustard gas",
    # WWII específico
    "Blitzkrieg",
    "Holocaust",
    "D-Day",
    "Normandy",
    "Nazi",
    "Wehrmacht",
    "Luftwaffe",
    "RAF",
    "Gestapo",
    "SS",
    "Pearl Harbor",
    "Hiroshima",
    "Stalingrad",
    "El Alamein",
    "Operation Overlord",
    "Battle of Berlin",
    # Batallas (ambas guerras)
    "Battle of Britain",
    "Battle of Midway",
    "Battle of Stalingrad",
    "Battle of the Bulge",
    # Tecnología y armamento
    "tank warfare",
    "Panzer",
    "air superiority",
    "strategic bombing",
    "fighter aircraft",
    "bomber aircraft",
    "radar",
    "codebreaking",
    "Enigma",
    # Consecuencias
    "Cold War",
    "United Nations",
    "post-war reconstruction",
    "war crimes",
    "Nuremberg Trials",
]


MAX_ARTICLES = 5000  # límite para mantener tiempos de embedding razonables


# Función para verificar si el título de un artículo contiene alguna de las palabras clave (case-insensitive)
def match_keyword(title, keywords):
    title = title.lower()
    for kw in keywords:
        pattern = r"\b" + re.escape(kw.lower()) + r"\b"
        if re.search(pattern, title):
            return True
    return False


print("Cargando dataset Wikipedia en modo streaming...")
wiki_stream = load_dataset(
    "wikimedia/wikipedia", "20231101.en", split="train", streaming=True
)

# Filtramos artículos relevantes por título usando las palabras clave definidas
articles = []
for example in wiki_stream:
    title = example["title"]
    # Filtramos por keywords en el título (case-insensitive)
    if match_keyword(title, WW_KEYWORDS):
        articles.append(
            {
                "title": title,
                "url": example["url"],
                "text": title + "\n\n" + example["text"],  # título + cuerpo
            }
        )
    if len(articles) >= MAX_ARTICLES:
        break

print(f"Artículos recuperados: {len(articles)}")
print("Muestra de títulos:")
for a in articles[:10]:
    print(f'  - {a["title"]}')

Cargando dataset Wikipedia en modo streaming...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

'HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00000-of-00041.parquet
Retrying in 1s [Retry 1/5].
'HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00000-of-00041.parquet
Retrying in 1s [Retry 1/5].
'HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00001-of-00041.parquet
Retrying in 1s [Retry 1/5].
'HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/re

KeyboardInterrupt: 

In [ ]:
# Convertimos a DataFrame para facilitar la exploración
pd.set_option("display.max_colwidth", 120)
df = pd.DataFrame(articles)

# Calculamos longitud en palabras de cada artículo
df["word_count"] = df["text"].apply(lambda t: len(t.split()))
df["char_count"] = df["text"].apply(len)

print(f"Shape del DataFrame: {df.shape}")
df[["title", "word_count", "char_count"]].head(25)

Shape del DataFrame: (5000, 5)


,title,word_count,char_count
0,2015 Pan American Cross Country Cup,161,1035
1,2015 ISSF World Cup,57,378
2,Ghaleb Husseini,540,3479
3,Ropica crassepuncta,29,193
4,Rafael Ferrer,22,164
5,Anne Rasmussen (educator),422,2834
6,Robert Award for Best Actress in a Supporting Television Role,87,553
7,List of ambassadors of Australia to Afghanistan,501,3351
8,Assassination of Harry Barrie,132,793
9,Geoglossum difforme,55,358


---
# 2. Análisis Exploratorio del Corpus (EDA)

En esta etapa se realiza un análisis detallado del corpus con el **propósito de fundamentar decisiones clave en el diseño del sistema RAG**.

En particular, se busca comprender la longitud y estructura de los artículos, lo cual permite definir un `chunk_size` adecuado para la segmentación del texto. Asimismo, se analiza la distribución temática del corpus para verificar el balance entre eventos de la Primera y Segunda Guerra Mundial, evitando sesgos en el proceso de recuperación.

Adicionalmente, se examinan los conceptos y términos predominantes con el fin de validar la relevancia semántica del corpus respecto al dominio de interés. Este análisis también permite identificar posibles vacíos o sobre-representaciones en la información disponible.

Estas decisiones son fundamentales para asegurar que el sistema de recuperación y generación opere de manera eficiente, maximizando la calidad del contexto proporcionado al modelo y, en consecuencia, la precisión de las respuestas generadas.

## EDA 1: Distribución de longitud de artículos

In [ ]:
sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de palabras
axes[0].hist(
    df["word_count"], bins=50, color="steelblue", edgecolor="white", alpha=0.85
)
axes[0].axvline(
    df["word_count"].mean(),
    color="red",
    linestyle="--",
    label=f'Media: {df["word_count"].mean():.0f}',
)
axes[0].axvline(
    df["word_count"].median(),
    color="orange",
    linestyle="--",
    label=f'Mediana: {df["word_count"].median():.0f}',
)
axes[0].set_title("Distribución de palabras por artículo")
axes[0].set_xlabel("Número de palabras")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

# Boxplot de caracteres
axes[1].boxplot(
    df["char_count"],
    vert=True,
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", alpha=0.6),
)
axes[1].set_title("Distribución de caracteres por artículo (boxplot)")
axes[1].set_ylabel("Número de caracteres")
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

# Estadísticas descriptivas clave
print("=== Estadísticas de longitud (palabras) ===")
print(f'  Media:       {df["word_count"].mean():.0f} palabras')
print(f'  Mediana:     {df["word_count"].median():.0f} palabras')
print(f'  Percentil 5: {df["word_count"].quantile(0.05):.0f} palabras')
print(f'  Percentil 25: {df["word_count"].quantile(0.25):.0f} palabras')
print(f'  Percentil 75: {df["word_count"].quantile(0.75):.0f} palabras')
print(f'  Percentil 95: {df["word_count"].quantile(0.95):.0f} palabras')
print(f'  Máximo:      {df["word_count"].max():.0f} palabras')

**Notas — Distribución de longitudes:**

La distribución es **asimétrica a la derecha**: la mayoría de artículos son cortos-medianos, pero existen artículos muy extensos (biografías, batallas principales) que sesgan la media hacia arriba.

El histograma confirma que la mayor densidad se concentra en artículos por debajo de ~1000 palabras, mientras que el boxplot de caracteres evidencia una gran cantidad de **outliers**, con artículos que superan ampliamente el rango intercuartílico. Esto sugiere una alta heterogeneidad en la longitud del corpus, típica de articulos Wikipedia.

**Implicación para el chunking:**

- Artículos cortos (≈ <500 palabras): existe riesgo de **sobre-fragmentación** si el `chunk_size` es muy pequeño, lo que puede romper coherencia semántica y afectar la calidad del embedding.
  
- Artículos medianos (≈ 500–1500 palabras): representan la mayoría del corpus, por lo que el `chunk_size` debe optimizarse principalmente para este rango.

- Artículos largos (>2000 palabras): requieren segmentación efectiva; chunks demasiado grandes pueden **mezclar múltiples subtemas**, reduciendo la precisión del retriever.

**Hipótesis inicial:** un `chunk_size ≈ 512 caracteres` ofrece un buen equilibrio entre coherencia semántica y granularidad, permitiendo capturar unidades informativas compactas sin diluir el contexto. Esta elección deberá validarse posteriormente mediante análisis de sensibilidad.

## EDA 2: Distribución por sub-tema (WWI, WWII, General)


In [ ]:
# Keywords más completas y menos ambiguas
WWI_KEYWORDS = [
    "world war i",
    "first world war",
    "wwi",
    "western front",
    "eastern front",
    "trench warfare",
    "armistice",
    "treaty of versailles",
    "verdun",
    "somme",
    "gallipoli",
    "austro-hungarian",
    "ottoman empire",
    "franz ferdinand",
]

WWII_KEYWORDS = [
    "world war ii",
    "second world war",
    "wwii",
    "nazi",
    "hitler",
    "holocaust",
    "d-day",
    "normandy landings",
    "blitzkrieg",
    "wehrmacht",
    "luftwaffe",
    "pearl harbor",
    "hiroshima",
    "nagasaki",
    "stalingrad",
    "el alamein",
    "operation barbarossa",
    "battle of berlin",
]


def keyword_match(text, keywords):
    text = text.lower()
    for kw in keywords:
        pattern = r"\b" + re.escape(kw) + r"\b"
        if re.search(pattern, text):
            return True
    return False


def classify_article(title):
    title_lower = title.lower()

    is_ww1 = keyword_match(title_lower, WWI_KEYWORDS)
    is_ww2 = keyword_match(title_lower, WWII_KEYWORDS)

    if is_ww1 and is_ww2:
        return "Ambas/General"
    elif is_ww1:
        return "WWI"
    elif is_ww2:
        return "WWII"
    else:
        return "General"


# Clasificación
df["theme"] = df["title"].apply(classify_article)

# Conteo
theme_counts = df["theme"].value_counts()

# Visualización
fig, ax = plt.subplots(figsize=(7, 7))

colors = {
    "WWI": "#4472C4",
    "WWII": "#ED7D31",
    "Ambas/General": "#A9D18E",
    "General": "#BFBFBF",
}

ax.pie(
    theme_counts.values,
    labels=theme_counts.index,
    autopct="%1.1f%%",
    colors=[colors.get(k, "#CCCCCC") for k in theme_counts.index],
    startangle=140,
    textprops={"fontsize": 12},
)

ax.set_title("Distribución del corpus por sub-tema", fontsize=14, pad=15)

plt.tight_layout()
plt.show()

# Output textual
print("Conteo por sub-tema:")
print(theme_counts.to_string())

## EDA 3: WordCloud de títulos 

In [ ]:
# Revela qué eventos, lugares y organizaciones dominan el corpus a nivel de título
stopwords_titles = STOPWORDS | {"of", "the", "in", "at", "on", "and", "Battle", "War"}
title_text = " ".join(df["title"].tolist())

# Configuración del WordCloud para títulos
wc_titles = WordCloud(
    width=900,
    height=450,
    background_color="white",
    stopwords=stopwords_titles,
    colormap="Blues",
    max_words=80,
    collocations=False,
).generate(title_text)

# Visualización del WordCloud de títulos
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(wc_titles, interpolation="bilinear")
ax.axis("off")
ax.set_title("WordCloud — Términos más frecuentes en títulos de artículos", fontsize=14)
plt.tight_layout()
plt.show()

## EDA 4: WordCloud del contenido

In [ ]:
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_stopwords

# Tomamos las primeras 300 palabras de cada artículo para acelerar el proceso
# Usamos stopwords de NLTK para filtrar términos vacíos

en_stopwords = set(nltk_stopwords.words("english")) | STOPWORDS

# Concatenamos extractos de todos los artículos
content_sample = " ".join(
    " ".join(text.split()[:300])  # primeras 300 palabras de cada artículo
    for text in df["text"].tolist()
)

wc_content = WordCloud(
    width=900,
    height=450,
    background_color="#1a1a2e",
    stopwords=en_stopwords,
    colormap="YlOrRd",
    max_words=80,
    collocations=False,
).generate(content_sample)

# Visualización del WordCloud de contenido
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(wc_content, interpolation="bilinear")
ax.axis("off")
ax.set_title("WordCloud — Conceptos dominantes en el contenido del corpus", fontsize=14)
plt.tight_layout()
plt.show()

Notas del EDA
----

El análisis exploratorio del corpus evidencia varios patrones relevantes que impactan directamente el diseño del sistema RAG.

En primer lugar, se observa un **desbalance temático significativo**, con una clara predominancia de artículos generales y, en menor medida, de la Segunda Guerra Mundial, mientras que la Primera Guerra Mundial está subrepresentada. Esto sugiere que el sistema de recuperación tenderá a responder con mayor precisión en consultas relacionadas con WWII o temas generales, lo cual debe considerarse como una limitación inherente del corpus.

En términos de longitud, los artículos presentan una **alta variabilidad**, con una concentración en rangos medios (≈ 600–1000 palabras) y la presencia de outliers considerablemente más extensos. Esto implica que el tamaño de los chunks debe balancear adecuadamente **coherencia semántica y granularidad**, evitando tanto la fragmentación excesiva como la mezcla de múltiples subtemas en un mismo segmento.

El análisis léxico refuerza la validez del corpus: los términos dominantes en el contenido (e.g., *German, British, French, forces, attack, troops*) indican una fuerte presencia de lenguaje militar y contextual relevante. Sin embargo, la WordCloud de títulos revela cierta **heterogeneidad temática**, incluyendo entidades no estrictamente bélicas (e.g., escuadrones, divisiones, nombres propios), lo que sugiere que el filtrado inicial introduce algo de ruido.

En conjunto, el corpus es suficientemente rico en semántica de dominio, pero presenta **desbalance y ruido estructural**, lo cual puede afectar la precisión del retriever en ciertos casos.

---

### Implicación para el chunking

Dado este comportamiento, se plantea como hipótesis inicial evaluar tres configuraciones de tamaño de chunk:

- `256` caracteres: mayor granularidad, pero riesgo de pérdida de contexto  
- `512` caracteres: equilibrio entre coherencia y precisión  
- `1024` caracteres: mayor contexto, pero posible dilución semántica  

Estas configuraciones serán evaluadas en la siguiente sección para determinar cuál optimiza la calidad de recuperación en el sistema.

---
# 3. Preprocesamiento y Análisis de Sensibilidad al Chunking

Los modelos de embedding tienen una ventana de tokens limitada (tipicamente 512 tokens para `e5-large`). Los artículos de Wikipedia suelen superar ese límite, por lo que **debemos dividirlos en fragmentos (chunks)** antes de indexarlos.

### Parámetros clave

- **`chunk_size`**: número máximo de caracteres por fragmento. Fragmentos pequeños → mayor granularidad, menor contexto. Fragmentos grandes → más contexto, menor especificidad.
- **`chunk_overlap`**: solapamiento entre fragmentos consecutivos. Evita cortar frases a la mitad y preserva continuidad temática.

`RecursiveCharacterTextSplitter`

LangChain ofrece este splitter que divide primero por párrafos (`\n\n`), luego por líneas (`\n`), y finalmente por espacios. Es el más robusto para texto narrativo como Wikipedia.

In [ ]:
from langchain_core.documents import Document

# Convertimos cada artículo a un objeto Document de LangChain
# Los metadatos (título, URL) se conservarán en cada chunk para citar la fuente
docs = [
    Document(page_content=a["text"], metadata={"title": a["title"], "url": a["url"]})
    for a in articles
]

print(f"Documentos creados: {len(docs)}")
print(f"Ejemplo de metadata: {docs[0].metadata}")
print(f"Longitud del primer documento: {len(docs[0].page_content)} chars")

## Prueba de Sensibilidad al Chunking

In [ ]:
# Definimos 3 configuraciones que abarcan el espacio de decisión relevante
CHUNK_CONFIGS = {
    "small": {"chunk_size": 256, "chunk_overlap": 32},  # fragmento corto
    "medium": {"chunk_size": 512, "chunk_overlap": 64},  # ~1 párrafo
    "large": {"chunk_size": 1024, "chunk_overlap": 128},  # ~2-3 párrafos
}

split_results = {}
for name, cfg in CHUNK_CONFIGS.items():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"],
        chunk_overlap=cfg["chunk_overlap"],
        # Separadores: párrafos → líneas → espacios → caracteres
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    lengths = [len(c.page_content) for c in chunks]
    split_results[name] = {
        "chunks": chunks,
        "lengths": lengths,
        "n": len(chunks),
        "avg": int(sum(lengths) / len(lengths)),
        "median": int(sorted(lengths)[len(lengths) // 2]),
        "min": min(lengths),
        "max": max(lengths),
    }

# Tabla resumen
print(
    f"{'Config':<10} {'N° chunks':>10} {'Avg (chars)':>12} {'Median':>8} {'Min':>6} {'Max':>8}"
)
print("-" * 58)
for name, r in split_results.items():
    print(
        f'{name:<10} {r["n"]:>10,} {r["avg"]:>12,} {r["median"]:>8,} {r["min"]:>6,} {r["max"]:>8,}'
    )

In [ ]:
# ── Visualización de las distribuciones de longitud por configuración ─────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
palette = {"small": "#4472C4", "medium": "#ED7D31", "large": "#70AD47"}

for ax, (name, r) in zip(axes, split_results.items()):
    ax.hist(r["lengths"], bins=40, color=palette[name], edgecolor="white", alpha=0.85)
    ax.axvline(
        r["avg"], color="red", linestyle="--", linewidth=1.5, label=f'Media: {r["avg"]}'
    )
    ax.axvline(
        r["median"],
        color="black",
        linestyle=":",
        linewidth=1.5,
        label=f'Mediana: {r["median"]}',
    )
    ax.set_title(
        f'chunk_size={CHUNK_CONFIGS[name]["chunk_size"]} ({name})\nN={r["n"]:,} chunks',
        fontsize=12,
    )
    ax.set_xlabel("Longitud del chunk (chars)")
    ax.set_ylabel("Frecuencia")
    ax.legend(fontsize=9)

plt.suptitle(
    "Distribución de longitud de chunks por configuración", fontsize=14, y=1.02
)
plt.tight_layout()
plt.show()

## Decisión de Chunking

A partir del EDA y el análisis de sensibilidad sobre la longitud de los chunks, se observa un comportamiento consistente: en las tres configuraciones, la distribución se sesga hacia el límite superior del `chunk_size`, lo que indica que el splitter está aprovechando al máximo la capacidad definida antes de fragmentar. Esto es deseable, ya que sugiere que los chunks contienen unidades semánticas relativamente completas.

| Config | Pros | Contras |
|---|---|---|
| **small** (256/32) | Alta granularidad, mayor precisión en hechos puntuales | Fragmentación excesiva, pérdida de coherencia semántica |
| **medium** (512/64) | Balance adecuado entre contexto y precisión | Ligera redundancia por overlap |
| **large** (1024/128) | Mayor contexto por chunk, útil para consultas complejas | Riesgo de mezclar múltiples subtemas y reducir precisión del retriever |

Adicionalmente, los histogramas muestran que:

- En **small**, la mediana (~176 chars) sugiere que muchos chunks quedan incompletos semánticamente.
- En **medium**, la mediana (~371 chars) se acerca más a unidades textuales coherentes.
- En **large**, aunque se maximiza el contexto (mediana ~782 chars), aumenta la probabilidad de introducir ruido semántico dentro de un mismo chunk.

---

### Elección final

Se selecciona **`medium` (512 chars / 64 overlap)** como configuración principal, ya que ofrece el mejor compromiso entre granularidad y coherencia.

Esta decisión se fundamenta en:

1. La estructura típica de los artículos de Wikipedia, donde los párrafos suelen concentrar información semánticamente consistente dentro de rangos intermedios de longitud.
2. La distribución observada, donde esta configuración evita tanto la sobre-fragmentación como la sobre-agregación de contenido.
3. La compatibilidad con el modelo de embeddings (`multilingual-e5-large`), donde 512 caracteres corresponden aproximadamente a ~100–130 tokens, manteniéndose cómodamente dentro del límite de entrada.
4. El uso de un `overlap` de 64 caracteres, que introduce redundancia controlada y mejora la continuidad entre fragmentos consecutivos, favoreciendo la recuperación de contexto relevante.

En conjunto, esta configuración maximiza la probabilidad de que cada chunk represente una unidad semántica coherente, lo cual es crítico para el desempeño del retriever en tareas de búsqueda y generación aumentada.

> La validación final se realizará de forma cualitativa en el sistema RAG, evaluando si los chunks recuperados mantienen coherencia y relevancia frente a distintas consultas.

In [ ]:
# Seleccionamos la configuración medium como la principal para el RAG
chunks = split_results["medium"]["chunks"]
print(f"Chunks seleccionados (medium): {len(chunks):,}")
print(f"Ejemplo de chunk:")
print(f"  Metadata: {chunks[0].metadata}")
print(f"  Contenido (primeros 300 chars): {chunks[0].page_content[:300]}...")

---
# 4. Configuración del LLM: Mistral 7B Instruct

| Característica | **mistral:7b-instruct** |
|---|---|
| Parámetros |  **7B** |
| Ventana de contexto |  **32K** |
| Benchmark MMLU |  **~64%** |
| Instruction-following | **Muy bueno** |
| Uso de VRAM (cuantizado) | **~4.1 GB** |

Mistral 7B Instruct v0.3 tiene mejor razonamiento y síntesis de texto, lo que se traduce en respuestas históricas más precisas y coherentes. Cabe dentro de la GPU T4 de Colab (16 GB VRAM) con cuantización Q4.

> **Nota:** Necesitamos que `ollama serve` esté corriendo antes de ejecutar las siguientes celdas.

In [ ]:
# Instalamos dependencias del sistema y Ollama si no está disponible
!sudo apt install zstd -y -q
!if ! type ollama > /dev/null 2>&1; then \
    echo 'Instalando Ollama...' && curl -fsSL https://ollama.com/install.sh | sh; \
else \
    echo 'Ollama ya está instalado.'; \
fi

In [ ]:
# Cargamos la extensión de terminal para Colab
# En la terminal que se abre, ejecuta: ollama serve &
# Esto lanza el servidor de Ollama en segundo plano
%load_ext colabxterm
%xterm

In [ ]:
# Descargamos el modelo Mistral 7B Instruct cuantizado (Q4 por defecto en Ollama)
# El modelo pesa ~4.1 GB — puede tardar 2-5 minutos en Colab con buena conexión
!ollama pull mistral:7b-instruct

In [ ]:
from langchain_ollama import ChatOllama

# temperature=0.25: respuestas casi deterministas, importantes para precisión histórica
# Un temperature alto generaría respuestas más 'creativas' pero menos fiables en RAG
llm = ChatOllama(model="mistral:7b-instruct", temperature=0.25)

# Prueba de sanity: verificamos que el LLM responde antes de continuar
response = llm.invoke("Who started World War I and what were the main causes?")
print("=== Respuesta del LLM (sin contexto RAG) ===")
print(response.content)

----

# 5. Creación del Vector Store con FAISS

### Pipeline de indexación

```
chunks (texto) → Embedding Model → vectores float32 → FAISS Index
```

1. **`intfloat/multilingual-e5-large`**: modelo de embedding de 560M parámetros, entrenado en 94 idiomas. Produce vectores de 1024 dimensiones. A pesar de ser multilingüe, mantiene excelente rendimiento en inglés puro (MTEB benchmark).

2. **FAISS** (Facebook AI Similarity Search): librería especializada en búsqueda de vecinos más cercanos en espacios vectoriales de alta dimensión. Usa índices aproximados para escalar a millones de vectores manteniendo baja latencia.

3. **Guardado del índice**: el índice FAISS se serializa a disco. En ejecuciones posteriores se carga directamente, evitando recalcular todos los embeddings (~30 min en T4 para 3,000 artículos).

In [ ]:
import os
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Inicializamos el modelo de embeddings con normalización L2 (recomendado para e5)
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    encode_kwargs={
        "normalize_embeddings": True
    },  # necesario para similitud coseno correcta
)

index_path = "./faiss_ww_index"

if os.path.exists(index_path):
    # Cargamos el índice existente (evita re-embeddear todo el corpus)
    print("Cargando índice FAISS existente...")
    vectorstore = FAISS.load_local(
        index_path, embeddings, allow_dangerous_deserialization=True
    )
else:
    # Construimos el índice desde los chunks — puede tardar 20-40 min en T4
    print(f"Construyendo índice FAISS para {len(chunks):,} chunks...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(index_path)
    print("Índice guardado en disco.")

# El retriever buscará los 5 chunks más similares a cada consulta (k=5)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print(f"Retriever listo. Índice contiene {vectorstore.index.ntotal:,} vectores.")

---
# 6. Pipeline RAG: Pregunta-Respuesta Simple



En esta sección se construye una cadena RAG básica que integra recuperación de información y generación de respuestas. El objetivo es evaluar de forma cualitativa cómo el sistema utiliza el contexto recuperado para responder preguntas, así como la coherencia y trazabilidad de las respuestas generadas.

El flujo sigue una arquitectura estándar donde, dada una pregunta, el retriever identifica los fragmentos más relevantes del corpus y estos son inyectados en el prompt del modelo para guiar la generación. Este enfoque permite que el modelo produzca respuestas fundamentadas en evidencia explícita, en lugar de depender únicamente de su conocimiento paramétrico.

Usamos `create_retrieval_chain` (forma moderna, no deprecada) que devuelve automáticamente el contexto recuperado como lista de `Document` objects, facilitando la cita de fuentes.

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

# Prompt en inglés para alinearlo con el corpus
# Instruimos al LLM a citar fuentes y admitir incertidumbre
prompt = PromptTemplate.from_template(
    """\
You are a knowledgeable historian specializing in World War I and World War II.
Use the following context fragments retrieved from Wikipedia to answer the question.
If the answer is not in the context, say so honestly — do not hallucinate.
Always include citations to the source articles at the end of your response.

Context:
{context}

Question: {input}
Answer:"""
)

# create_stuff_documents_chain concatena todos los chunks en el campo {context}
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
# create_retrieval_chain une el retriever con la cadena de documentos
qa_chain = create_retrieval_chain(retriever, combine_docs_chain)


def format_answer(result):
    """Formatea la respuesta incluyendo título y URL de cada fuente recuperada."""
    answer = result["answer"] + "\n\n**Fuentes:**\n"
    seen = set()
    for i, doc in enumerate(result["context"], 1):
        title = doc.metadata.get("title", "Desconocido")
        url = doc.metadata.get("url", "")
        key = title  # deduplicamos por título
        if key not in seen:
            answer += f"  [{i}] {title} — {url}\n"
            seen.add(key)
    return answer


print("Cadena RAG configurada correctamente.")

In [ ]:
# Pruebas con preguntas históricas
test_questions = [
    "What were the main causes of World War I?",
    "Describe the D-Day invasion and its significance.",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {q}")
    print("=" * 60)
    result = qa_chain.invoke({"input": q})
    print(format_answer(result))

### Observaciones sobre la calidad del RAG simple

- **Precisión de retrieval**: Las fuentes recuperadas deben corresponder temáticamente a la pregunta. Si aparecen artículos irrelevantes, podría indicar que `k=5` es demasiado alto o que el corpus tiene ruido.
- **Calidad de síntesis**: Mistral 7B Instruct organiza bien la información de múltiples fragmentos en una respuesta coherente.
- **Limitación del RAG simple**: cada pregunta es independiente — si haces preguntas de seguimiento ('¿Y el general que lo lideró?'), el modelo no tiene memoria de la pregunta anterior. Esto lo solucionamos en la siguiente sección.

---
# 7. Cadena Conversacional con Historial

El problema del RAG sin memoria
---

En el QA simple, cada invocación es independiente. Si el usuario pregunta:
1. *'Tell me about the Battle of Stalingrad'*
2. *'Who commanded the German forces there?'*

La segunda pregunta introduce una referencia implícita (*"there"*) que depende del contexto previo. Sin acceso al historial, el retriever no puede inferir que se refiere a Stalingrado, lo que degrada la calidad de la recuperación y, en consecuencia, de la respuesta generada.

---

Solución: `create_history_aware_retriever`
---

Para abordar esta limitación, se introduce un retriever consciente del historial que incorpora una etapa previa de reformulación. En este enfoque, el modelo utiliza el historial conversacional junto con la nueva pregunta para generar una versión explícita y autónoma de la consulta.

El flujo es el siguiente:

historial + nueva pregunta  
↓  
el LLM reformula la pregunta (ej. *"Who commanded the German forces at Stalingrad?"*)  
↓  
retriever  
↓  
LLM genera respuesta  

De esta forma, el sistema transforma preguntas ambiguas en consultas completas antes de la recuperación, mejorando significativamente la relevancia de los documentos recuperados y la coherencia de las respuestas en contextos conversacionales.

In [ ]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

# Prompt para condensar el historial + nueva pregunta en una pregunta autónoma
condense_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given the conversation history and the latest user question, "
            "reformulate the question into a standalone question that can be understood "
            "without the conversation context. Keep the original language. "
            "Do NOT answer the question — only reformulate it if needed.",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)

# El retriever ahora recibe la pregunta condensada
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, condense_prompt
)

# Prompt principal del QA conversacional
qa_system_prompt = (
    "You are a knowledgeable historian specializing in World War I and World War II. "
    "Use the following retrieved context to answer the question. "
    "If you cannot find the answer in the context, say so. "
    "Be concise but informative.\n\n{context}"
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)

# Cadena final: retriever consciente del historial + generación
qa_chain_conv = create_stuff_documents_chain(llm, qa_prompt)
convo_qa_chain = create_retrieval_chain(history_aware_retriever, qa_chain_conv)

print("Cadena conversacional configurada.")

## Demostración de conversación multi-turno 

In [ ]:
# Mostramos cómo el historial permite preguntas de seguimiento con referencias implícitas

chat_history = []

conversation = [
    "Tell me about the Battle of Stalingrad.",
    "Who commanded the German forces there?",  # 'there' = Stalingrad
    "What happened to him after the battle?",  # 'him' = von Paulus
]

for question in conversation:
    print(f"\n{'─'*60}")
    print(f"Usuario: {question}")
    print("─" * 60)

    response = convo_qa_chain.invoke(
        {
            "input": question,
            "chat_history": chat_history,
        }
    )

    # Actualizamos el historial para la siguiente vuelta
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response["answer"]))

    print(f'Asistente: {response["answer"]}')
    # Mostramos las fuentes recuperadas
    sources = list({d.metadata["title"] for d in response["context"]})
    print(f"[Fuentes: {', '.join(sources[:3])}]")

---
# 8. Interfaz de Usuario — ChatBot Gradio



Construimos la interfaz conversacional con **Gradio Blocks**, que ofrece más flexibilidad que `gr.ChatInterface`.

### Gestión dual de historiales

| Historial | Tipo | Propósito |
|---|---|---|
| `lc_history` | `List[HumanMessage \| AIMessage]` | Pasado a LangChain en cada invocación |
| `chat_history` (Gradio) | `List[Tuple[str, str]]` | Renderizado visual en el componente `gr.Chatbot` |

Ambos historiales se resetean al presionar el botón **Limpiar**, garantizando que una nueva conversación no herede contexto de la anterior.

In [ ]:
import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage

# Historial de LangChain (estructuras internas)
lc_history = []


def respond(question, chat_history):
    """Procesa la pregunta del usuario y actualiza ambos historiales."""
    if not question.strip():
        return "", chat_history

    # Invocamos la cadena conversacional con el historial de LangChain
    reply = convo_qa_chain.invoke({"input": question, "chat_history": lc_history})

    # Actualizamos el historial interno de LangChain
    lc_history.append(HumanMessage(content=question))
    lc_history.append(AIMessage(content=reply["answer"]))

    # Formateamos respuesta con fuentes para la UI
    answer = format_answer(reply)

    # Actualizamos el historial visual de Gradio
    chat_history.append((question, answer))
    return "", chat_history


def reset_chat():
    """Limpia ambos historiales para iniciar una conversación nueva."""
    lc_history.clear()  # vaciamos el historial de LangChain
    return [], ""  # vaciamos la UI de Gradio


# Construimos la interfaz
with gr.Blocks(title="WW Historian RAG", theme=gr.themes.Soft()) as gr_blocks:
    gr.Markdown(
        """
    # Historiador de las Guerras Mundiales — RAG
    Pregunta sobre la **Primera** o **Segunda Guerra Mundial**.
    Las respuestas están fundamentadas en artículos de Wikipedia y el modelo **Mistral 7B Instruct**.
    """
    )

    chatbot = gr.Chatbot(label="Conversación", height=450)
    msg = gr.Textbox(
        label="Tu pregunta",
        placeholder="Ej: What were the main battles of World War II?",
        lines=2,
    )

    with gr.Row():
        submit_btn = gr.Button("Enviar", variant="primary")
        clear_btn = gr.Button("Limpiar conversación")

    # Enviamos con Enter o con el botón
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    submit_btn.click(respond, [msg, chatbot], [msg, chatbot])
    clear_btn.click(reset_chat, None, [chatbot, msg], queue=False)

gr_blocks.launch(inline=False, share=IN_COLAB)

In [ ]:
# Cerramos la interfaz de Gradio al finalizar el taller
gr_blocks.close()

---
# 9. Conclusiones

### Resumen del taller

Implementamos un **sistema RAG completo** especializado en la historia de las Guerras Mundiales, recorriendo todo el pipeline:

1. **Curaduría del corpus**: Filtrado semántico de Wikipedia para obtener un corpus temáticamente relevante.
2. **EDA informada**: El análisis de la distribución de longitudes y la composición temática del corpus fundamentó directamente las decisiones de chunking.
3. **Chunking con análisis de sensibilidad**: Comparamos tres configuraciones y justificamos `chunk_size=512` como balance óptimo entre granularidad y contexto.
4. **Indexación vectorial con FAISS**: Usamos embeddings multilinguales de alta calidad y búsqueda por similitud coseno.
5. **Cadena conversacional**: Incorporamos historial de conversación para preguntas de seguimiento naturales.
6. **Interfaz Gradio**: Desplegamos el chatbot con gestión correcta de los dos historiales (LangChain + Gradio).

### Tabla comparativa: configuraciones de chunking

| Config | N° chunks | Granularidad | Contexto por chunk | Recomendado para |
|---|---|---|---|---|
| small (256/32) | Mayor | Alta | Bajo | Preguntas de hechos muy concretos |
| **medium (512/64)** | **Medio** | **Media** | **Medio** | **Uso general (elegido)** |
| large (1024/128) | Menor | Baja | Alto | Preguntas de análisis y síntesis |

### Análisis crítico — Limitaciones

- **Desbalance del corpus**: el corpus tiene más artículos de WWII que de WWI debido a los keywords de filtrado; esto puede afectar la calidad del retrieval para preguntas específicas de WWI.
- **Latencia**: Mistral 7B con Ollama en Colab T4 tarda ~5-15 segundos por respuesta. En producción se usaría un servicio de inferencia optimizado (vLLM, TGI).
- **Evaluación cualitativa**: no implementamos métricas automáticas de evaluación RAG (ej. RAGAS, que mide faithfulness, answer relevancy, context precision). Esto es una mejora directa para el siguiente paso.
- **Chunking fijo**: usamos `RecursiveCharacterTextSplitter` con un tamaño fijo. Enfoques más avanzados como *semantic chunking* (dividir por cambios de tema detectados con embeddings) podrían mejorar la coherencia de cada chunk.

### Conexión con la teoría del curso

- **Transformers y embeddings** (Unidad 4): el modelo `e5-large` es un Transformer encoder fine-tuneado con contrastive learning para producir representaciones semánticas densas. La similitud coseno en el espacio de embeddings implementa la noción de *semantic similarity* vista en clase.
- **Generación de texto** (Unidad 5): Mistral 7B Instruct es un Transformer decoder con instruction-tuning, similar al GPT estudiado, pero con mejoras como Grouped Query Attention (GQA) y Sliding Window Attention (SWA).
- **RAG vs. fine-tuning**: RAG actualiza el conocimiento del sistema sin reentrenar; el fine-tuning adapta los pesos del modelo. Para dominios que cambian frecuentemente (noticias, documentos legales), RAG es la arquitectura más práctica.

### Posibles mejoras

- **Evaluación automática**: integrar [RAGAS](https://github.com/explodinggradients/ragas) para medir faithfulness y relevancy de las respuestas.
- **Búsqueda híbrida**: combinar similitud vectorial (FAISS) con BM25 (keyword matching) usando `EnsembleRetriever` de LangChain.
- **Re-ranking**: aplicar un modelo cross-encoder (ej. `ms-marco-MiniLM`) para reordenar los chunks recuperados antes de pasarlos al LLM.
- **Semantic chunking**: reemplazar `RecursiveCharacterTextSplitter` por `SemanticChunker` de LangChain Experimental.